In [ ]:
import pandas as pd
import requests
import zipfile
import io

# 1. Download the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
df = pd.read_csv(z.open('SMSSpamCollection'), sep='\t', names=['label', 'message'])

# 2. Encode labels: ham = 0, spam = 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Dataset Loaded: {df.shape[0]} messages found.")
df.head()

Dataset Loaded: 5572 messages found.


,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase and remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    # Tokenize and remove stopwords
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df['clean_msg'] = df['message'].apply(clean_text)
print("Sample Cleaned Text:", df['clean_msg'].iloc[0])

Sample Cleaned Text: go jurong point crazy available bugis n great world la e buffet cine got amore wat


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize Vectorizer
tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(df['clean_msg']).toarray()
y = df['label_num']

print(f"Feature Matrix Shape: {X.shape}") # Should be (5572, 1000)

Feature Matrix Shape: (5572, 1000)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear SVM
# We use linear because we can export the 'weights' easily for Flutter
model = SVC(kernel='linear')
model.fit(X_train, y_train)

# Show Metrics
y_pred = model.predict(X_test)
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

--- Confusion Matrix ---
[[964   2]
 [ 14 135]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.91      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [ ]:
import json
import numpy as np

# 1. Save the Vocabulary (Word -> Index mapping)
# Convert NumPy int64 values to standard Python int
vocab = {str(k): int(v) for k, v in tfidf.vocabulary_.items()}

with open('tfidf_vocab.json', 'w') as f:
    json.dump(vocab, f)

# 2. Save SVM Weights (The Hyperplane coordinates)
# Check if weights are sparse or dense, then convert to list
if hasattr(model.coef_, "toarray"):
    weights = model.coef_.toarray()[0].tolist()
else:
    weights = model.coef_[0].tolist()

bias = float(model.intercept_[0])

svm_data = {
    "weights": weights,
    "bias": bias
}

with open('svm_weights.json', 'w') as f:
    json.dump(svm_data, f, indent=4)

print("✅ Success! 'tfidf_vocab.json' and 'svm_weights.json' are ready for download.")

✅ Success! 'tfidf_vocab.json' and 'svm_weights.json' are ready for download.
